# Hybrid Deepfake Detector — Local GPU Execution

This notebook runs the **Hybrid Deepfake Detector** pipeline end-to-end natively on your local machine using GPU acceleration (NVIDIA Quadro P6000 / CUDA).

**Pipeline Architecture & Stages:**
1. **Local GPU Diagnostics & Workspace Setup** (VRAM check, folder structure)
2. **Kaggle Authentication & Dataset Download** (Celeb-DF v2 + DFD via `kagglehub`)
3. **Metadata Generation & Stratified Identity-Aware Splits** (prevents identity leakage across train/val/test)
4. **Feature Precomputation** (MTCNN face crops, Farneback optical flow sequences, rPPG + frequency forensic features)
5. **Branch Training**:
   - Semantic Branch: Fine-tuned EfficientNet-B0 with differential LR & class weights
   - Temporal Branch: ResNet-18 + LSTM sequence classifier
6. **Multi-Modal Embedding Extraction** (575-D fused vector per video)
7. **Distribution Matching & RBF SVM Fusion Classification**
8. **Final Test-Set Evaluation & Metrics Report**
9. **Ablation Study** (Individual vs Combined Branch Performance)
10. **Strict Cross-Dataset Generalization** (Train on DFD -> Test on Celeb-DF v2)

> **Note:** All precomputed features and checkpoints are cached locally in `./data` and `./saved_models`. You can stop and resume at any stage.

## 1. Local Environment & GPU Diagnostics

In [ ]:
import os
import sys
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    cuda_cap = torch.cuda.get_device_capability(0)
    print(f"Active GPU: {gpu_name}")
    print(f"Total VRAM: {total_mem:.2f} GB")
    print(f"Compute Capability: {cuda_cap}")
else:
    print("WARNING: Running on CPU. GPU acceleration is highly recommended.")

## 2. Setup Local Workspace Directories

In [ ]:
workspace_dirs = [
    "data/raw",
    "data/processed",
    "data/metadata",
    "data/splits",
    "models",
    "saved_models"
]

for path in workspace_dirs:
    os.makedirs(path, exist_ok=True)
    print(f"Verified directory: {os.path.abspath(path)}")

## 3. Kaggle API Authentication

Ensure your `kaggle.json` API token is located at `~/.kaggle/kaggle.json` (or enter your credentials below if not already set).

In [ ]:
import json
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_json_path = kaggle_dir / "kaggle.json"

if not kaggle_json_path.exists():
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    print(f"Kaggle credentials not found at {kaggle_json_path}.")
    print("To configure manually, uncomment and fill the lines below:")
    # creds = {"username": "YOUR_KAGGLE_USERNAME", "key": "YOUR_KAGGLE_API_KEY"}
    # with open(kaggle_json_path, "w") as f:
    #     json.dump(creds, f)
    # print("Credentials saved successfully.")
else:
    with open(kaggle_json_path, "r") as f:
        data = json.load(f)
        username = data.get("username", "unknown")
    print(f"Kaggle credentials found for user: {username}")

## 4. Download Datasets (Celeb-DF v2 & DFD)

Downloads are handled via `kagglehub` and organized automatically into `data/raw/`.

In [ ]:
from src.data.adapters.celebdf import CelebDFAdapter
from src.data.adapters.dfd import DFDAdapter

print("Downloading / Verifying Celeb-DF v2...")
CelebDFAdapter().download()

print("Downloading / Verifying DFD (Google/Jigsaw dataset)...")
DFDAdapter().download()

print("Dataset verification complete.")

## 5. Generate Metadata & Create Balanced Identity-Aware Splits

In [ ]:
print("--- 1. Scanning and generating full metadata ---")
!python -m src.data.metadata

print("\n--- 2. Subsetting to balanced targets (1000 real / 1000 fake per dataset) ---")
!python -m src.preprocessing.subset_metadata --target celebdf:1000:1000 --target dfd:1000:1000

print("\n--- 3. Creating leakage-safe identity-aware splits ---")
!python -m src.preprocessing.create_splits

## 6. Precompute Multi-Modal Features

Precomputes face crops, optical flow sequences, and rPPG/frequency forensic features (cached under `data/processed/`, fully resumable).

In [ ]:
print("--- [1/3] Precomputing Semantic Face Crops (MTCNN) ---")
!python -m src.preprocessing.precompute_faces

print("\n--- [2/3] Precomputing Temporal Sequences (Farneback Optical Flow) ---")
!python -m src.preprocessing.precompute_temporal

print("\n--- [3/3] Precomputing Forensic Features (rPPG + FFT / Wavelet) ---")
!python -m src.preprocessing.precompute_forensic

## 7. Train Semantic Branch (Fine-Tuned EfficientNet-B0)

Trains with class-weighted CrossEntropyLoss and validation Macro-F1 model checkpointing.

In [ ]:
!python -m src.modeling.train_semantic

## 8. Train Temporal Branch (ResNet-18 + LSTM)

Trains the temporal feature extractor and LSTM sequence model on 16-frame optical flow face sequences.

In [ ]:
!python -m src.modeling.train_temporal

## 9. Extract Embeddings (Semantic 256-D + Temporal 256-D + Forensic 63-D = 575-D)

In [ ]:
!python -m src.modeling.extract_embeddings

## 10. Train Fusion Classifier (Distribution Matching + RBF SVM)

In [ ]:
!python -m src.modeling.train_fusion

## 11. Final Test-Set Evaluation (Headline Result)

In [ ]:
!python -m src.modeling.test_fusion

In [ ]:
# Load and display the evaluation report
import json
import pandas as pd

report_path = "saved_models/test_fusion_evaluation_report.json"
if os.path.exists(report_path):
    with open(report_path, "r") as f:
        report = json.load(f)
    print("\n=== Headline Test Evaluation Summary ===")
    for k, v in report.items():
        if isinstance(v, float):
            print(f"{k:25s}: {v:.4f}")
        elif not isinstance(v, (dict, list)):
            print(f"{k:25s}: {v}")
else:
    print(f"Report not found at {report_path}. Run Section 11 first.")

## 12. Ablation Study (Multi-Modal Branch Contribution)

In [ ]:
!python -m src.modeling.run_ablation

## 13. Cross-Dataset Generalization Experiment (Train on DFD, Test on Celeb-DF v2)

In [ ]:
# 1. Create cross-dataset split
!python -m src.preprocessing.create_cross_dataset_splits --train-dataset dfd --test-dataset celebdf --out-dir data/splits_cross_dfd_to_celebdf

# 2. Extract embeddings for cross-dataset split
!python -m src.modeling.extract_embeddings --splits-root data/splits_cross_dfd_to_celebdf --output-root data/processed/fusion_cross_dfd_to_celebdf

# 3. Train fusion model on DFD only
!python -m src.modeling.train_fusion --embeddings-root data/processed/fusion_cross_dfd_to_celebdf --model-path models/fusion_classifier/fusion_model_cross_dfd_to_celebdf.pkl --report-path saved_models/train_fusion_report_cross_dfd_to_celebdf.json

# 4. Evaluate strictly on unseen Celeb-DF v2
!python -m src.modeling.test_fusion --embeddings-root data/processed/fusion_cross_dfd_to_celebdf --model-path models/fusion_classifier/fusion_model_cross_dfd_to_celebdf.pkl --report-path saved_models/test_fusion_evaluation_report_cross_dfd_to_celebdf.json

## 14. Archive Results and Artifacts

In [ ]:
import shutil
import glob

shutil.make_archive("results_bundle", "zip", "saved_models")
print("Created results archive: results_bundle.zip")

print("\nSaved Reports:")
for f in glob.glob("saved_models/*.json"):
    print(f"  - {f}")

print("\nSaved Checkpoints:")
for f in glob.glob("saved_models/*.pth") + glob.glob("models/**/*.pkl", recursive=True):
    print(f"  - {f}")